In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

TONE_CENTROID_CSV = DATA_DIR / "tone_centroid_embeddings.csv"
CLUSTER_DEF_PATH  = DATA_DIR / "brand_tone_cluster.csv"

OUT_FIXED_CSV = DATA_DIR / "tone_centroid_embeddings_fixed.csv"
OUT_FIXED_NPY = DATA_DIR / "tone_centroid_embeddings_fixed.npy"

if not TONE_CENTROID_CSV.exists():
    raise FileNotFoundError(f"없음: {TONE_CENTROID_CSV}")
if not CLUSTER_DEF_PATH.exists():
    raise FileNotFoundError(f"없음: {CLUSTER_DEF_PATH}")

cluster_def_df = pd.read_csv(CLUSTER_DEF_PATH)
cluster_names = cluster_def_df["brand"].astype(str).str.strip().tolist()
K = len(cluster_names)

raw = pd.read_csv(TONE_CENTROID_CSV)
print("[INFO] raw cols head:", list(raw.columns)[:12])

# 0~767 숫자 컬럼만 추출 (문자열 메타 컬럼 제외)
vec_cols = [c for c in raw.columns if str(c).strip().isdigit()]
vec_cols = sorted(vec_cols, key=lambda x: int(x))

if len(vec_cols) == 0:
    raise ValueError("벡터 컬럼(0,1,2,...)을 못 찾음. 파일이 centroid 벡터가 맞는지 확인 필요.")
print("[INFO] vec_cols:", vec_cols[:5], "...", vec_cols[-5:], "count=", len(vec_cols))

X = raw[vec_cols].to_numpy(dtype=np.float32)
print("[INFO] X shape:", X.shape)

if X.shape[0] != K:
    raise ValueError(f"centroid row({X.shape[0]}) != cluster_names({K})")

fixed = pd.DataFrame(X, columns=[str(i) for i in range(X.shape[1])])
fixed.insert(0, "tone_id", cluster_names)

fixed.to_csv(OUT_FIXED_CSV, index=False, encoding="utf-8-sig")
np.save(OUT_FIXED_NPY, X)

print("=" * 60)
print("[DONE] fixed centroid saved")
print("CSV:", OUT_FIXED_CSV)
print("NPY:", OUT_FIXED_NPY)
print("shape:", X.shape)
print("=" * 60)

display(fixed.head())

[INFO] raw cols head: ['brand', 'brand_tone_cluster', 'brand_position', '0', '1', '2', '3', '4', '5', '6', '7', '8']
[INFO] vec_cols: ['0', '1', '2', '3', '4'] ... ['763', '764', '765', '766', '767'] count= 768
[INFO] X shape: (12, 768)
[DONE] fixed centroid saved
CSV: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27/data/tone_centroid_embeddings_fixed.csv
NPY: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27/data/tone_centroid_embeddings_fixed.npy
shape: (12, 768)


,tone_id,0,1,2,3,4,5,6,7,8,...,758,759,760,761,762,763,764,765,766,767
0,Clinical,-0.884826,-0.870754,-4.579100,-1.204269,0.788411,-0.663912,0.347561,-0.818273,-0.757952,...,1.693823,-0.160747,-0.189353,-0.988368,0.306682,-0.048281,-0.876696,-0.100946,-0.657813,-1.061333
1,Dermatological,-1.174374,-0.557896,-4.601971,-1.360816,0.791214,-0.675256,0.281267,-0.592000,-0.433251,...,1.730731,-0.271144,-0.017763,-1.062377,0.397834,0.110465,-0.801439,0.071741,-0.623446,-1.078222
2,Ingredient-Driven,-0.755689,-0.710913,-4.280115,-1.263458,1.067544,-0.565830,0.370630,-0.418440,-0.611535,...,1.869585,-0.257051,-0.307285,-1.131516,0.288903,0.185312,-0.827542,-0.153439,-0.577134,-0.913299
3,Empathetic,-1.174374,-0.557896,-4.601971,-1.360816,0.791214,-0.675256,0.281267,-0.592000,-0.433251,...,1.730731,-0.271144,-0.017763,-1.062377,0.397834,0.110465,-0.801439,0.071741,-0.623446,-1.078222
4,Care-Oriented,-0.755689,-0.710913,-4.280115,-1.263458,1.067544,-0.565830,0.370630,-0.418440,-0.611535,...,1.869585,-0.257051,-0.307285,-1.131516,0.288903,0.185312,-0.827542,-0.153439,-0.577134,-0.913299


In [15]:
# ============================================================
# Brand → Tone Cluster Mapping (REBUILD)
# - 실행 위치: third_week/12_27
# - 입력:
#   data/brand_analysis_part_enhanced.csv
#   data/tone_centroid_embeddings.npy
# - 출력:
#   data/brand_tone_cluster_by_brand.csv
# ============================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

# ============================================================
# 경로 (CWD = third_week/12_27 기준)
# ============================================================

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

BRAND_PART_PATH = DATA_DIR / "brand_analysis_part_enhanced.csv"
TONE_CENTROID_PATH = DATA_DIR / "tone_centroid_embeddings.npy"

OUT_PATH = DATA_DIR / "brand_tone_cluster_by_brand.csv"

# ============================================================
# 유틸
# ============================================================

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = s.replace("\u00a0", " ")
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s


def parse_embedding(val, dim):
    if isinstance(val, (list, np.ndarray)):
        arr = np.array(val, dtype=np.float32).ravel()
    else:
        nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", str(val))
        arr = np.array(nums, dtype=np.float32) if nums else np.zeros(dim, dtype=np.float32)

    if arr.size < dim:
        arr = np.pad(arr, (0, dim - arr.size))
    elif arr.size > dim:
        arr = arr[:dim]

    return arr


# ============================================================
# 로드
# ============================================================

for p in [BRAND_PART_PATH, TONE_CENTROID_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"파일 없음: {p}")

brand_part = pd.read_csv(BRAND_PART_PATH)
tone_centroids = np.load(TONE_CENTROID_PATH).astype(np.float32)

print("[INFO] brand_part cols:", list(brand_part.columns))
print("[INFO] tone_centroids:", tone_centroids.shape)

# 컬럼 정리
if "브랜드" in brand_part.columns and "brand" not in brand_part.columns:
    brand_part = brand_part.rename(columns={"브랜드": "brand"})

if "embedding_vector" not in brand_part.columns:
    raise ValueError("brand_analysis_part_enhanced.csv에 embedding_vector 컬럼 없음")

brand_part["brand"] = brand_part["brand"].map(norm)

dim = tone_centroids.shape[1]

# ============================================================
# 브랜드별 평균 임베딩 계산
# ============================================================

rows = []

for brand, g in brand_part.groupby("brand"):
    vecs = []
    for v in g["embedding_vector"]:
        arr = parse_embedding(v, dim)
        if not np.all(arr == 0):
            vecs.append(arr)

    if not vecs:
        continue

    brand_vec = np.mean(vecs, axis=0).reshape(1, -1)

    sims = cosine_similarity(brand_vec, tone_centroids)[0]
    best_cluster = int(np.argmax(sims))
    best_score = float(np.max(sims))

    rows.append({
        "brand": brand,
        "brand_tone_cluster": best_cluster,
        "confidence": round(best_score, 4)
    })

# ============================================================
# 저장
# ============================================================

out_df = pd.DataFrame(rows).sort_values("brand")

out_df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")

print("=" * 60)
print("brand → tone cluster 매핑 완료")
print("brands:", len(out_df))
print("PATH:", OUT_PATH)
print("=" * 60)

[INFO] brand_part cols: ['brand', 'part_id', 'content', '핵심 키워드', 'AI 분석 톤', '확신도', '형용사 수', 'embedding_vector', 'char_len', 'part_role']
[INFO] tone_centroids: (12, 768)
brand → tone cluster 매핑 완료
brands: 30
PATH: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27/data/brand_tone_cluster_by_brand.csv


In [13]:
from pathlib import Path

print("CWD:", Path.cwd())
print("DATA_DIR:", (Path.cwd() / "data"))
print("Exists?", (Path.cwd() / "data" / "brand_analysis_part_enhanced.csv").exists())

CWD: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27
DATA_DIR: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27/data
Exists? True
